# Milestone 2: Visualization Prototyping

This notebook prototypes the interactive visualizations that will become the core of our web application. The design aligns with the professor's feedback from Milestone 1:
1. Highlighting the influence of political context on cyberattacks.
2. Evaluating how Switzerland compares to other target nations.
3. Translating the global threat into the localized reality of Kanton Zürich.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

eurepoc = pd.read_csv('../Dataset/eurepoc_global_dataset_1_3.csv')
ktzh = pd.read_csv('../Dataset/KTZH_00001202_00003680.csv')

eurepoc['start_date'] = pd.to_datetime(eurepoc['start_date'], dayfirst=True, errors='coerce')

## Graph 1: The Geopolitical Streamgraph

**Goal:** Provide a narrative centerpiece that explains the macro context of global cyber warfare.  
**Analysis:** Instead of relying on a simple line chart, this **Streamgraph** elegantly solves two problems at once: it shows the massive explosion in overall cyber incident volume over time, while simultaneously breaking down *what kind* of attacks those were. By mapping key political triggers over this structure, we can see if events like the Ukraine Invasion triggered specific kinds of warfare (like Disruption vs. Espionage).

In [2]:
eurepoc['year_month'] = eurepoc['start_date'].dt.to_period('M').dt.to_timestamp()

events = [
    {"date": "2020-03-01", "label": "COVID-19 Pandemic Declared (Mar 2020)", "short": "COVID-19 Pandemic"},
    {"date": "2020-12-01", "label": "SolarWinds Hack (Dec 2020)", "short": "SolarWinds Breach"},
    {"date": "2021-05-01", "label": "Colonial Pipeline (May 2021)", "short": "Colonial Pipeline"},
    {"date": "2022-02-01", "label": "Invasion of Ukraine (Feb 2022)", "short": "Invasion of Ukraine"},
    {"date": "2023-10-01", "label": "Israel-Hamas Conflict (Oct 2023)", "short": "Israel-Hamas Conflict"}
]


def map_attack_type(inc_type):
    inc_type = str(inc_type).lower()
    
    if 'ddos' in inc_type or 'disrupt' in inc_type or 'defacement' in inc_type:
        return 'Disruption'
    elif 'ransomware' in inc_type or 'malware' in inc_type or 'hijack' in inc_type or 'exploit' in inc_type:
        return 'Exploitation'
    elif 'phish' in inc_type or 'social' in inc_type or 'theft' in inc_type or 'info' in inc_type or 'leak' in inc_type:
        return 'Info-Ops'
        
    return 'Others'

eurepoc['Aggregated_Attack_Type'] = eurepoc['incident_type'].apply(map_attack_type)

color_mapping = {
    'Disruption': '#1f77b4',   
    'Exploitation': '#d62728', 
    'Info-Ops': '#2ca02c',     
    'Others': '#7f7f7f',       
    
    'Cyber Fraud': '#d62728',         
    'Defamation': '#2ca02c',          
    'Money & Package Mules': '#7f7f7f', 
    'Sexual Offences': '#ff7f0e'      
}

category_order = ['Disruption', 'Exploitation', 'Info-Ops', 'Others']

TITLE_FONT_SIZE = 18


stream_data = eurepoc[eurepoc['year_month'] >= "2018-01-01"]


stream_data = stream_data.groupby(['year_month', 'Aggregated_Attack_Type']).size().unstack(fill_value=0).stack().reset_index(name='incidents')

stream_data['monthly_total'] = stream_data.groupby('year_month')['incidents'].transform('sum')
# Avoid division by zero by using a small safe fallback
stream_data['percentage'] = stream_data.apply(
    lambda row: (row['incidents'] / row['monthly_total'] * 100) if row['monthly_total'] > 0 else 0,
    axis=1
)

fig1 = px.area(
    stream_data, 
    x='year_month', 
    y='incidents', 
    color='Aggregated_Attack_Type',
    color_discrete_map=color_mapping,
    category_orders={'Aggregated_Attack_Type': category_order},
    title='Graph 1: Global Cyber Incident Categories vs Geopolitics (Streamgraph)',
    labels={'year_month': 'Time', 'incidents': 'Total Volume', 'Aggregated_Attack_Type': 'Attack Category'},
    custom_data=['Aggregated_Attack_Type', 'percentage']
)

fig1.update_traces(
    hovertemplate="<b>%{customdata[0]}</b>: %{customdata[1]:.1f}%<br>Incidents: %{y}<extra></extra>"
)

for evt in events:
    fig1.add_vline(x=evt["date"], line_width=2, line_dash="dash", line_color="#111", opacity=0.7)
    fig1.add_annotation(
        x=evt["date"], y=stream_data.groupby('year_month')['incidents'].sum().max() * 0.95,
        text=evt["label"], showarrow=False, textangle=-90,
        xanchor='right', yanchor='top', font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,0.8)"
    )

fig1.update_layout(
    template='plotly_white', 
    hovermode='x unified',
    title_font_size=TITLE_FONT_SIZE
)
fig1.show()

### 🔎 Analysis of Graph 1
By plotting the raw incident data against verified geopolitical timelines, we uncover massive structural shifts in global cyber warfare. We can clearly map the *intent* of cyber operations by analyzing the top attack vectors alongside political events:

1. **The "January Spike" Artifact:** You may notice severe, isolated peaks at the start of certain years (e.g., January 2019 and January 2020). A deeper look into the dataset reveals that nearly 100% of incidents in those single-month spikes share an exact `start_date` of January 1st. This is a classic data reporting artifact where researchers default a log to `01.01.YYYY` when the exact date is unknown. We must consciously isolate this artifact from actual world events.

2. **The COVID-19 Paradigm Shift (Early 2020):** While the January spike is fake, the *sustained rise* starting in March 2020 is very real. When the **COVID-19 Pandemic** triggered global lockdowns, the attack surface exponentially widened due to hastily deployed remote-work infrastructure and unsecured VPNs. In 2020/2021, the overwhelming #1 attack type globally was **"Data theft & Hijacking with Misuse"**. State actors and cybercriminals exploited this chaos exclusively for espionage and financial gain, flawlessly illustrated by the **SolarWinds Breach** (Dec 2020) and the **Colonial Pipeline** ransomware (May 2021). The red *Exploitation* band clearly dominates the streamgraph during this prolonged pandemic era.

3. **The Tactical Warfare Shift (2022 Onwards) - Ukraine & Israel-Hamas:** Following the February 2022 **Russian invasion of Ukraine**, the global baseline permanently shifted from stealth to sabotage. 
   - While data theft remained prevalent, attacks categorized as **"Disruption" (DDoS, defacements, and wiper malware)** exploded from nearly zero to the #2 overall attack type globally in 2022 (67 incidents) and skyrocketed to over 114 incidents in 2023.
   - The Streamgraph visually proves this tactical shift: the blue **Disruption** band instantly widens the moment tanks cross into Ukraine, driven by state-aligned hacktivists (e.g., Killnet, IT Army of Ukraine) paralyzing critical infrastructure. 
   - This exact blueprint of utilizing *Disruption* as an auxiliary weapon to kinetic warfare repeated itself perfectly during the late 2023 **Israel-Hamas** conflict, confirming that modern geopolitical skirmishes now universally spawn parallel cyber-sabotage campaigns.

## Graph 2: The Swiss Cyber-Landscape vs. Primary Targets

**Goal:** Address the feedback on comparing Switzerland's incident load and profile against other countries.  
**Analysis:** Instead of just comparing raw numbers (where the US dwarfs everyone), we will compare the *proportion* of different attack types directed at Switzerland, Ukraine, and the US. Do state actors hit Switzerland differently than they hit Ukraine?

In [3]:
def tag_target(row_val, target_names):
    if pd.isna(row_val): return False
    return any(name in str(row_val) for name in target_names)

eurepoc['target_CH'] = eurepoc['receiver_country'].apply(lambda x: tag_target(x, ['Switzerland']))
eurepoc['target_UA'] = eurepoc['receiver_country'].apply(lambda x: tag_target(x, ['Ukraine']))
eurepoc['target_US'] = eurepoc['receiver_country'].apply(lambda x: tag_target(x, ['United States']))

def get_profile(target_col, label):
    df_filtered = eurepoc[eurepoc[target_col] == True]
    counts = df_filtered['Aggregated_Attack_Type'].value_counts(normalize=True).reset_index()
    counts.columns = ['Aggregated_Attack_Type', 'percentage']
    counts['country'] = label
    return counts

profile_CH = get_profile('target_CH', 'Switzerland')
profile_UA = get_profile('target_UA', 'Ukraine')
profile_US = get_profile('target_US', 'United States')

global_avg = eurepoc['Aggregated_Attack_Type'].value_counts(normalize=True).reset_index()
global_avg.columns = ['Aggregated_Attack_Type', 'percentage']
global_avg['country'] = 'Global Average'

profiles = pd.concat([profile_CH, profile_UA, profile_US, global_avg])

fig2 = px.bar(
    profiles, 
    x='country', 
    y='percentage', 
    color='Aggregated_Attack_Type',
    color_discrete_map=color_mapping,
    category_orders={'Aggregated_Attack_Type': category_order},
    title='Attack Profile Comparison: Switzerland vs. Highly Targeted Nations',
    barmode='group',
    labels={
        'percentage': 'Proportion of Total Attacks (%)', 
        'country': 'Target Group',
        'Aggregated_Attack_Type': 'Attack Category'
    },
    text_auto='.1%',
    custom_data=['Aggregated_Attack_Type', 'percentage']
)

fig2.update_traces(
    hovertemplate="<b>%{customdata[0]}</b>: %{customdata[1]:.1%}<extra></extra>"
)

fig2.update_layout(
    template='plotly_white', 
    yaxis_tickformat='.0%',
    title_font_size=TITLE_FONT_SIZE
)
fig2.show()

### 🔎 Analysis of Graph 2: The Attack Profile Benchmark

**1. The Disruption Heavyweight (Switzerland)**
Surprisingly, Switzerland shows the highest proportion of **Disruption** attacks among the individual nations studied, at **52.2%**. This is significantly higher than the Global Average of **43.2%**. While Switzerland is often viewed as a "safe" or neutral zone, this figure suggests that Swiss infrastructure is a major target for service-denial and disruption tactics, likely aimed at its highly digitized financial and administrative sectors.

**2. The Exploitation Epicenter (Ukraine)**
Ukraine’s profile is dominated by **Exploitation** at **54.0%**, the highest in the comparison. In the context of the ongoing conflict, this high percentage indicates a massive focus on data theft, system compromise, and persistence. While Ukraine still faces a high level of **Disruption (40.9%)**, the strategic priority for attackers appears to be the exploitation of networks for intelligence or long-term compromise.

**3. The Information Operations Target (United States)**
The United States stands out for its high exposure to **Info-Ops**, which accounts for **17.9%** of its profile. This is nearly double the rate seen in Switzerland **(8.7%)** and triple that of Ukraine **(5.1%)**. This data proves that the US is a primary battlefield for influence campaigns, disinformation, and psychological operations, while its **Disruption** rate **(36.2%)** is the lowest of the group, suggesting attackers there favor "softer" or more covert methods of interference.

**4. Benchmarking Against the Global Average**
When compared to the **Global Average**, Switzerland over-indexes on **Disruption (52.2% vs 43.2%)** but under-indexes on **Info-Ops (8.7% vs 12.2%)**. This suggests that the Swiss "Cyber-Landscape" is more vulnerable to, or more frequently targeted by, direct technical interference rather than the broad influence operations that characterize global cyber warfare trends.


## Graph 3: The Local Reality (Zurich Deep-Dive)

**Goal:** Transition from global geopolitical hacking into local cybercrime reality (scams/fraud targeting citizens).  
**Analysis:** A normalized breakdown of Kanton Zürich's massive 6.5x growth in cyber offences over the last 8 years, focusing purely on what impacts the average resident (Financial Fraud vs Sexual Offences).

In [4]:
translation_map = {
    'digitalisierte Vermögenskriminalität': 'Cyber Fraud',
    'Cyber-Rufschädigung und unlauteres Verhalten': 'Defamation',
    'Cyber-Sexualdelikte': 'Sexual Offences',
    'Cybercrime (im engeren Sinne)': 'Core Cybercrime',
    'Darknet/Data Leaking': 'Darknet / Data Leaks'
}

ktzh['CyberModus_Überkategorie'] = ktzh['CyberModus_Überkategorie'].replace(translation_map)

local_trends = ktzh.groupby(['Ausgangsjahr', 'CyberModus_Überkategorie'])['Straftaten_total'].sum().reset_index()

local_trends['total_year'] = local_trends.groupby('Ausgangsjahr')['Straftaten_total'].transform('sum')
local_trends['percentage'] = (local_trends['Straftaten_total'] / local_trends['total_year']) * 100

local_order = ['Cyber Fraud', 'Defamation', 'Sexual Offences', 'Core Cybercrime', 'Darknet / Data Leaks']

local_color_mapping = {
    'Cyber Fraud': '#fa8072',          
    'Defamation': '#ffb6ff',           
    'Sexual Offences': '#ffe699',      
    'Core Cybercrime': '#9999ff',      
    'Darknet / Data Leaks': '#66cdaa'  
}

fig3 = px.area(
    local_trends, 
    x='Ausgangsjahr', 
    y='percentage', 
    color='CyberModus_Überkategorie',
    color_discrete_map=local_color_mapping,
    category_orders={'CyberModus_Überkategorie': local_order},
    title='Graph 3: Evolution of Local Cybercrime in Kanton Zürich (2016-2024)',
    labels={
        'Ausgangsjahr': 'Year', 
        'percentage': 'Proportion of Offences (%)', 
        'CyberModus_Überkategorie': 'Crime Category'
    },
    custom_data=['CyberModus_Überkategorie', 'percentage', 'Straftaten_total']
)

fig3.update_traces(
    hovertemplate="<b>%{customdata[0]}</b>: %{customdata[1]:.1f}%<br>Raw Count: %{customdata[2]}<extra></extra>"
)

annotations = []
last_year = local_trends['Ausgangsjahr'].max()
last_year_data = local_trends[local_trends['Ausgangsjahr'] == last_year].set_index('CyberModus_Überkategorie')

cumulative_y = 0
for cat in local_order:
    if cat in last_year_data.index:
        val = last_year_data.loc[cat, 'percentage']
        if pd.notnull(val) and val > 0:
            if cat == 'Cyber Fraud':
                mid_y = cumulative_y + (val / 2)
                annotations.append(
                    dict(
                        x=last_year - 0.1,  
                        y=mid_y,
                        text=f"<b>{cat}</b>",
                        showarrow=False,
                        xanchor="right",
                        font=dict(color="white", size=15)
                    )
                )
            cumulative_y += val

fig3.update_layout(
    template='plotly_white', 
    hovermode='x unified',
    showlegend=True,              
    legend_title_text='Crime Category',
    margin=dict(r=150),           
    annotations=annotations,      
    title_font_size=TITLE_FONT_SIZE,
    xaxis=dict(
        tickmode='array',
        tickvals=[2016, 2018, 2020, 2022, 2024],
        range=[local_trends['Ausgangsjahr'].min(), local_trends['Ausgangsjahr'].max()]
    ),
    yaxis=dict(ticksuffix="%")
)

fig3.show()

### 🔎 Analysis of Graph 3: The Local Impact Breakdown

When bringing the lens from global warfare down to the local level of Kanton Zürich, a stark contrast in "cyber" definitions emerges:

1. **The Supremacy of Cyber Fraud:** While nation-states worry about infrastructure disruption (Graph 1 & 2), the local citizen's reality is overwhelmingly dominated by **Cyber Fraud**. By the final recorded year, Cyber Fraud constitutes an astonishing **83.6%** of all local cyber offences. Despite its proportional dominance remaining relatively flat over time, the underlying raw volume is exploding, resulting in over **11,844** incidents alone.
2. **Core Cybercrime vs. Personal Harm:** "Core Cybercrime" (which includes traditional hacking, malware, and DDoS) accounts for only **11.3%** of offences reported to the local police (**1,594** raw incidents). In contrast, crimes affecting individuals directly, such as **Sexual Offences (4.6%, 649 incidents)**, represent a smaller but intensely critical segment of police effort that requires entirely different mitigation strategies.
3. **The Divergence of Scale:** The 100% stacked area chart elegantly demonstrates that the *composition* of cybercrime targeting Zürich residents hasn't radically shifted in flavor over the last decade. Instead, criminals have industrialized their attacks; they are perpetrating the exact same types of scams and fraud, but doing it **6.5x** more frequently than they did in 2016.

In [5]:
# Créer le dossier si nécessaire
import os
os.makedirs('../website/public/data', exist_ok=True)

# Exporter les figures en JSON
fig1.write_json('../website/public/data/graph1.json')
fig2.write_json('../website/public/data/graph2.json')
fig3.write_json('../website/public/data/graph3.json')

## Mapping Country to Intensity

In [2]:
# Split semicolon-separated countries and remove duplicates per row
eurepoc_expanded = eurepoc.copy()
eurepoc_expanded['receiver_country'] = eurepoc_expanded['receiver_country'].str.split(';')
# Remove duplicates within each list, then deduplicate
eurepoc_expanded['receiver_country'] = eurepoc_expanded['receiver_country'].apply(
    lambda x: list(set([c.strip() for c in x if pd.notna(c)]))
)
eurepoc_expanded = eurepoc_expanded.explode('receiver_country')

# Group by cleaned country and count incidents
country_to_intensity = eurepoc_expanded.groupby('receiver_country').size().reset_index(name='incident_count')
country_to_intensity = country_to_intensity.sort_values('incident_count', ascending=False)

print(f"Total unique countries: {len(country_to_intensity)}")
print(country_to_intensity.head(10))

Total unique countries: 205
       receiver_country  incident_count
194       United States             871
152              Russia             220
63              Germany             171
189      United Kingdom             165
136       Not available             154
87               Israel             149
59               France             138
187             Ukraine             137
80                India             134
94   Korea, Republic of             109


In [4]:
import json

# Load GeoJSON and extract country names
with open('../website/public/data/countries-50m.json', 'r') as f:
    geojson = json.load(f)

geo_countries = set()
for feature in geojson['objects']['countries']['geometries']:
    if 'properties' in feature and 'name' in feature['properties']:
        geo_countries.add(feature['properties']['name'])

print("=== GeoJSON Country Names (first 20) ===")
print(sorted(geo_countries)[:20])
print(f"Total unique countries in GeoJSON: {len(geo_countries)}\n")

# Extract unique country names from dataset
dataset_countries = set()
for val in eurepoc['receiver_country'].dropna():
    countries = val.split(';')
    for c in countries:
        dataset_countries.add(c.strip())

print("=== Dataset Country Names (first 20) ===")
print(sorted(dataset_countries)[:20])
print(f"Total unique countries in dataset: {len(dataset_countries)}\n")

# Find mismatches
in_dataset_not_geo = sorted(dataset_countries - geo_countries)
in_geo_not_dataset = sorted(geo_countries - dataset_countries)

print("=== In DATASET but NOT in GeoJSON ===")
print(in_dataset_not_geo)
print(f"Count: {len(in_dataset_not_geo)}\n")

print("=== In GeoJSON but NOT in dataset ===")
print(in_geo_not_dataset[:20])
print(f"Count: {len(in_geo_not_dataset)}")

=== GeoJSON Country Names (first 20) ===
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antarctica', 'Antigua and Barb.', 'Argentina', 'Armenia', 'Aruba', 'Ashmore and Cartier Is.', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados']
Total unique countries in GeoJSON: 241

=== Dataset Country Names (first 20) ===
['Afghanistan', 'Africa', 'Albania', 'Algeria', 'Angola', 'Anguilla', 'Argentina', 'Armenia', 'Asia (region)', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Balkans (region)', 'Bangladesh', 'Belarus', 'Belgium', 'Bermuda', 'Bhutan']
Total unique countries in dataset: 205

=== In DATASET but NOT in GeoJSON ===
['Africa', 'Asia (region)', 'Balkans (region)', 'Bosnia and Herzegovina', 'Caucasus', 'Central America (region)', 'Central Asia (region)', 'Congo, the Democratic Republic of the', 'Czech Republic', 'Dominican Republic', 'EU (institutions)', 'EU (region)', 'Eastern Asia (region)', '

In [8]:
# ============================================================================
# HELPER FUNCTIONS FOR COUNTRY DATA CLEANING
# ============================================================================

def split_and_expand_column(df, column_name):
    """
    Split semicolon-separated values in a column and expand rows.
    Removes duplicates within each cell before expanding.
    """
    df_expanded = df.copy()
    df_expanded[column_name] = df_expanded[column_name].str.split(';')
    df_expanded[column_name] = df_expanded[column_name].apply(
        lambda x: list(set([c.strip() for c in x if pd.notna(c)]))
    )
    return df_expanded.explode(column_name)

def clean_country_column(df, column_name, non_country_filters, country_mapping):
    """
    Filter out non-country entries and apply country name mappings.
    """
    df_clean = df.copy()
    # Filter out non-countries (regions, organizations, data quality issues)
    df_clean = df_clean[~df_clean[column_name].isin(non_country_filters)]
    # Apply country name mappings to match GeoJSON standards
    df_clean[column_name] = df_clean[column_name].replace(country_mapping)
    return df_clean

def build_nested_counts(df, outer_col, inner_col, count_col_name='count'):
    """
    Build nested dictionary structure: {outer_col: {inner_col: count, ...}, ...}
    Sorts inner dictionaries by count (descending).
    """
    grouped = df.groupby([outer_col, inner_col]).size().reset_index(name=count_col_name)
    
    nested = {}
    for _, row in grouped.iterrows():
        outer = row[outer_col]
        inner = row[inner_col]
        count = int(row[count_col_name])
        
        if outer not in nested:
            nested[outer] = {}
        nested[outer][inner] = count
    
    # Sort inner dictionaries by count (descending)
    for key in nested:
        nested[key] = dict(sorted(nested[key].items(), key=lambda x: x[1], reverse=True))
    
    return nested


In [9]:
# Clean country data using helper function
eurepoc_cleaned = clean_country_column(eurepoc_expanded, 'receiver_country', NON_COUNTRY_FILTERS, COUNTRY_MAPPING)

# Count incidents per country
country_to_intensity_clean = eurepoc_cleaned.groupby('receiver_country').size().reset_index(name='incident_count')
country_to_intensity_clean = country_to_intensity_clean.sort_values('incident_count', ascending=False)

print(f"✓ Removed {len(eurepoc_expanded) - len(eurepoc_cleaned)} non-country/organization/region entries")
print(f"✓ Valid countries after cleaning: {len(country_to_intensity_clean)}")
print(f"\nTop 20 countries by incident count:")
print(country_to_intensity_clean.head(20).to_string())

# Export cleaned mapping
country_intensity_dict_clean = dict(zip(country_to_intensity_clean['receiver_country'], country_to_intensity_clean['incident_count']))

import os
os.makedirs('../website/public/data', exist_ok=True)

with open('../website/public/data/country-intensity.json', 'w') as f:
    json.dump(country_intensity_dict_clean, f, indent=2)

print("\n✓ Exported cleaned country-intensity.json")


✓ Removed 532 non-country/organization/region entries
✓ Valid countries after cleaning: 165

Top 20 countries by incident count:
             receiver_country  incident_count
157  United States of America             871
124                    Russia             220
53                    Germany             171
156            United Kingdom             165
71                     Israel             149
50                     France             138
154                   Ukraine             137
66                      India             134
135               South Korea             109
28                     Canada              92
30                      China              91
151                    Turkey              89
73                      Japan              88
72                      Italy              81
68                       Iran              73
112                  Pakistan              71
7                   Australia              71
143                    Taiwan              

## Mapping Country to Source Attribution

**Enhancement:** Instead of just counting incidents per country, we now track the *source* (initiator country) of incidents targeting each country. This reveals geopolitical attack patterns.

**Output Structure:**
```json
{
  "United States": {"Russia": 15, "China": 8, "Unknown": 42, ...},
  "Ukraine": {"Russia": 23, "Unknown": 5, ...},
  ...
}
```


In [11]:
# Expand both receiver and initiator countries to handle semicolon-separated values
eurepoc_source_data = split_and_expand_column(eurepoc_expanded.copy(), 'receiver_country')
eurepoc_source_data = split_and_expand_column(eurepoc_source_data, 'initiator_country')

# Clean both receiver and initiator countries
eurepoc_source_data = clean_country_column(eurepoc_source_data, 'receiver_country', NON_COUNTRY_FILTERS, COUNTRY_MAPPING)
eurepoc_source_data = clean_country_column(eurepoc_source_data, 'initiator_country', NON_COUNTRY_FILTERS, COUNTRY_MAPPING)

# Build nested structure using helper function
country_to_sources = build_nested_counts(eurepoc_source_data, 'receiver_country', 'initiator_country')

# Export to JSON
with open('../website/public/data/country-sources.json', 'w') as f:
    json.dump(country_to_sources, f, indent=2)

print("✓ Created nested country-to-sources mapping")
print(f"✓ Total countries: {len(country_to_sources)}")
print(f"\nExample - Top incident sources targeting United States:")
if 'United States of America' in country_to_sources:
    usa_sources = country_to_sources['United States of America']
    for source, count in list(usa_sources.items())[:5]:
        print(f"  {source}: {count} incidents")

print(f"\nExample - Top incident sources targeting Ukraine:")
if 'Ukraine' in country_to_sources:
    ua_sources = country_to_sources['Ukraine']
    for source, count in list(ua_sources.items())[:5]:
        print(f"  {source}: {count} incidents")

print("\n✓ Exported country-sources.json")


✓ Created nested country-to-sources mapping
✓ Total countries: 153

Example - Top incident sources targeting United States:
  Not attributed: 289 incidents
  China: 114 incidents
  Russia: 79 incidents
  Iran: 46 incidents
  North Korea: 28 incidents

Example - Top incident sources targeting Ukraine:
  Russia: 80 incidents
  Not attributed: 20 incidents
  China: 6 incidents
  Ukraine: 6 incidents
  Belarus, Russia: 1 incidents

✓ Exported country-sources.json
